# 02 - Parameter sampling by ParamSampler

This notebook confirms that `pipelines.tuning_pipeline.tuners.ParamSampler`
translates a search-space dictionary into concrete suggestions on an Optuna
trial, and that the three sampling kinds behave as declared. It exercises
`ParamSampler.sample` against real Optuna trials driven by a `RandomSampler`
so the empirical distributions are easy to read.

We confirm: float parameters land inside their range; log floats are uniform in
log space; categorical parameters cover all choices; and `indexed_categorical`
stores an integer `__idx` on the trial while returning the decoded value.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## Drive the sampler with a random study

We collect the values the sampler returns and the raw parameters Optuna records
on each trial. The two differ for `indexed_categorical`, where Optuna stores an
index suffixed with `__idx`.

In [ ]:
from pipelines.tuning_pipeline.tuners import ParamSampler
from configuration.models_config import UNetConfig

sampler    = ParamSampler()
lr_space   = UNetConfig.tunable_lr_params()
arch_space = UNetConfig.tunable_arch_params()
space      = {**lr_space, **arch_space}

returned = []
raw      = []

def collect(trial):
    returned.append(sampler.sample(trial, space))
    raw.append(dict(trial.params))
    return 0.0

study = optuna.create_study(sampler=optuna.samplers.RandomSampler(seed=SEED))
study.optimize(collect, n_trials=400)

print('returned keys:', sorted(returned[0].keys()))
print('raw keys     :', sorted(raw[0].keys()))

## Float parameters stay inside their declared range

Histograms of the sampled learning rate (log-spaced) and dropout (linear)
confirm the values respect the configured bounds and the requested spacing.

In [ ]:
enc_lr  = np.array([r['encoder_lr'] for r in returned])
dropout = np.array([r['dropout']    for r in returned])

fig, (a0, a1) = plt.subplots(1, 2, figsize=(10, 4))

a0.hist(np.log10(enc_lr), bins=25, color='steelblue', edgecolor='white')
a0.axvline(np.log10(lr_space['encoder_lr']['low']),  color='k', ls='--', lw=1)
a0.axvline(np.log10(lr_space['encoder_lr']['high']), color='k', ls='--', lw=1)
a0.set_xlabel('log10(encoder_lr)')
a0.set_ylabel('count')
a0.set_title('log-spaced float (flat in log)')

a1.hist(dropout, bins=25, color='indianred', edgecolor='white')
a1.axvline(lr_space['dropout']['low'],  color='k', ls='--', lw=1)
a1.axvline(lr_space['dropout']['high'], color='k', ls='--', lw=1)
a1.set_xlabel('dropout')
a1.set_title('linear float (flat in value)')
fig.tight_layout()
plt.show()

print('encoder_lr  in range:', enc_lr.min()  >= lr_space['encoder_lr']['low'] and enc_lr.max()  <= lr_space['encoder_lr']['high'])
print('dropout     in range:', dropout.min() >= lr_space['dropout']['low']    and dropout.max() <= lr_space['dropout']['high'])

## Categorical coverage

Counting the sampled values of an ordinary categorical parameter confirms every
declared choice is reachable.

In [ ]:
from collections import Counter

act_counts = Counter(r['activation'] for r in returned)
choices    = arch_space['activation']['choices']

fig, ax = plt.subplots()
ax.bar(choices, [act_counts.get(c, 0) for c in choices], color='seagreen')
ax.set_ylabel('count')
ax.set_title("categorical 'activation' coverage over 400 trials")
fig.tight_layout()
plt.show()

print('all activation choices sampled:', all(act_counts.get(c, 0) > 0 for c in choices))

## Indexed categorical: raw index versus decoded value

For `features` the sampler suggests an integer index (`features__idx`) on the
trial but returns the decoded list-of-channels. We confirm the decoded value
always equals `choices[raw_index]`, which is exactly what the orchestrator's
`_decode_phase1_best` relies on.

In [ ]:
feat_choices = arch_space['features']['choices']

consistent = True
idx_values = []
for ret, rw in zip(returned, raw):
    idx = rw['features__idx']
    idx_values.append(idx)
    consistent &= (ret['features'] == feat_choices[idx])

print('decoded == choices[raw_idx] for all trials:', consistent)

fig, ax = plt.subplots()
ax.hist(idx_values, bins=np.arange(len(feat_choices) + 1) - 0.5, color='slateblue', edgecolor='white', rwidth=0.9)
ax.set_xticks(range(len(feat_choices)))
ax.set_xticklabels([str(c) for c in feat_choices], rotation=20, fontsize=8)
ax.set_ylabel('count')
ax.set_xlabel('features__idx -> decoded choice')
ax.set_title('indexed_categorical sampling')
fig.tight_layout()
plt.show()

## Expected visual outcome

The log-spaced learning rate forms a roughly flat histogram between the dashed
bounds, while dropout is flat in linear value. Both range checks print `True`.
Every activation choice receives a non-zero bar, and the indexed-categorical
consistency check prints `True` with all feature indices represented.